# ⚡ HELIOS AI COGNITIVE RUNTIME — Cloud GPU Pretraining Accelerator
### **High-Throughput Pretraining (120k–180k tok/s), Zero-Loss Checkpointing & HF/Ollama Sync**

**Architect & Creator**: Arron Kian Parejas — Creator of Helios & Domodomo · Top 1 on AppBuilders · Former GDG Chapter Lead (Philippines)  

---

### **Key Hardware & Storage Accelerations**
1. **High-Throughput GPU (H100 / A100 / L4)**: Native **bfloat16 precision** + `torch.compile(mode="reduce-overhead")` + PyTorch SDPA kernel fusion.
2. **Continuous Google Drive Auto-Save**: Direct streaming to `/content/drive/MyDrive/Helios/`.
3. **Compute-Unit Preemption Shield**: Traps `SIGTERM`/`SIGINT`. Immediate checkpoint flush on disconnect.
4. **Resilient Auto-Downloader**: Automatically pulls pretraining corpus and latest verified weights from Hugging Face if Google Drive was initially empty!
5. **0% Duplicate Dataset Guarantee**: Cryptographic SHA-256 hash checking against `curriculum_ledger.json` ensures zero trained records are ever repeated.
6. **Live Hugging Face Hub Sync**: Automatic model upload to `Darknecrocities/Helios`.

## 1. Verify NVIDIA GPU Hardware & FlashAttention / BF16

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model: {gpu_name}")
    print(f"Total VRAM: {vram:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")
    print(f"TF32 Allowed: {torch.backends.cuda.matmul.allow_tf32}")

## 2. Mount Google Drive for Zero-Loss Checkpoint Persistence

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Helios')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'checkpoints').mkdir(exist_ok=True)
(DRIVE_ROOT / 'manifests').mkdir(exist_ok=True)
(DRIVE_ROOT / 'datasets').mkdir(exist_ok=True)
print(f"Google Drive Helios Root Ready: {DRIVE_ROOT}")

## 3. Pull Helios Repository & Set Hugging Face Credentials

In [ ]:
import os
%cd /content
!git clone -b pretrained https://github.com/darknecrocities/Helios.git helios_repo || (cd helios_repo && git pull origin pretrained)
%cd /content/helios_repo
!pip install -q -e .
!pip install -q safetensors rich tokenizers datasets huggingface_hub

# Set your HF_TOKEN from Colab Secrets (or pass environment variable)
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except Exception:
    pass
print("Helios environment initialized.")

## 4. Download / Verify Corpus & Checkpoints from Hugging Face Hub
If Google Drive is empty, this cell downloads the verified 383MB corpus and latest checkpoint weights directly into `/content/drive/MyDrive/Helios/` so pretraining resumes with zero duplicates.

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

hf_token = os.environ.get("HF_TOKEN")
corpus_target = DRIVE_ROOT / 'datasets' / 'helios_massive_pretrain_corpus.jsonl'
ledger_target = DRIVE_ROOT / 'manifests' / 'curriculum_ledger.json'

if not corpus_target.exists():
    print("Downloading massive pretraining corpus (120,616 records) from Hugging Face Hub...")
    corpus_dl = hf_hub_download(
        repo_id='Darknecrocities/Helios-Pretrain-Corpus',
        filename='helios_massive_pretrain_corpus.jsonl',
        repo_type='dataset',
        token=hf_token,
    )
    shutil.copy2(corpus_dl, corpus_target)
    print("Corpus ready at:", corpus_target)

if not ledger_target.exists():
    print("Downloading curriculum ledger from Hugging Face Hub...")
    ledger_dl = hf_hub_download(
        repo_id='Darknecrocities/Helios-Pretrain-Corpus',
        filename='curriculum_ledger.json',
        repo_type='dataset',
        token=hf_token,
    )
    shutil.copy2(ledger_dl, ledger_target)
    print("Ledger ready at:", ledger_target)

# Check status
import json
with open(ledger_target, 'r') as f:
    ledger_data = json.load(f)
print(f"\nLedger Cursor: {ledger_data.get('category_cursors', {}).get('all', 0)}")
print(f"Seen Hashes (0% Duplicate Guarantee): {len(ledger_data.get('seen_hashes', []))}")

## 5. Launch Pretraining Runner (2B Tokens Target)
Executes with bfloat16 mixed precision, TorchDynamo kernel fusion, Google Drive auto-checkpointing, and Hugging Face Hub sync.

In [ ]:
!python scripts/colab_runner.py \
    --drive-root "/content/drive/MyDrive/Helios" \
    --target-tokens 2000000000 \
    --batch-size 32 \
    --block-size 512

## 6. Interactive Verification & Deductive Reasoning Test

In [ ]:
!python -c "
from helios.cli.fallback import get_grounded_fallback
queries = [
    'who is napoleon?',
    'who is napoleon bonaparte?',
    'what is europe?',
    'who is donald trump?',
    'what is photosynthesis?'
]
for q in queries:
    ans = get_grounded_fallback(q, tier='nano-20m')
    print(f'\n[Query]: {q}')
    print(f'[Helios Response]: {ans}')
"